# 01 · Build and inspect the pilot sample

Gold Rails pilot: 250 rows across content (input, output, over-refusal) and prompt attacks (jailbreak, injection, leakage), drawn from seven ungated public sources. This notebook is the reference for how a sample is assembled; the same code runs from the CLI as `uv run python -m goldrails_dataset.build`.

Everything here is reproducible from a clean checkout with `uv sync --extra notebooks`.

In [ ]:
from pathlib import Path
import json, collections
import pandas as pd
from goldrails_dataset.build import build, write
from goldrails_dataset.records import read_jsonl

OUT = Path("../samples/pilot")
records = build(per_cell=25, seed=7)
manifest = write(records, OUT)
print(json.dumps(manifest, indent=2))

## What is in it

One row per (feature, subtask, expected, split). Sources are mixed inside a cell where the plan lists several, so a cell is never one source's idea of a label.

In [ ]:
rows = [r for f in OUT.glob("*.jsonl") for r in read_jsonl(f)]
df = pd.DataFrame([{
    "feature": r.feature, "subtask": r.subtask, "expected": r.expected, "split": r.split,
    "source": r.provenance.source, "label_basis": r.provenance.label_basis,
    "ailuminate": r.category.ailuminate, "bedrock": r.category.bedrock,
    "contaminated": bool(r.provenance.contamination), "chars": len(r.state.text),
} for r in rows])
display(df.groupby(["feature","subtask","expected","split"]).size().unstack("split").fillna(0).astype(int))
display(df.groupby(["feature","subtask","source"]).size().unstack("source").fillna(0).astype(int))

## Label provenance and contamination

`label_basis` other than `human` or `deterministic` is excluded from any headline later. `contamination` lists the guard-model evals or training sets a source is known to appear in; those rows form the anchor slice.

In [ ]:
display(df.groupby(["source","label_basis"]).size().unstack(fill_value=0))
display(df.groupby("source")["contaminated"].mean().rename("share_flagged").to_frame())
df["chars"].describe().to_frame().T

## Three rows, verbatim

A record carries the text, its role, any prior turn, both taxonomy labels, and provenance. It never carries a question; question sets are separate.

In [ ]:
for r in [rows[0], next(x for x in rows if x.subtask=="output"), next(x for x in rows if x.subtask=="leakage")]:
    print(json.dumps(r.to_dict(), indent=1, ensure_ascii=False)[:900], "\n---")